In [ ]:
# Ensure you have at least 2GB free space in Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())


In [ ]:
# Install dependencies if not already installed
!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image wandb iterative-stratification torchmetrics


In [ ]:
# Run this everytime you update something in the repo!
import os

REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

# if project directory is empty clone the repo, otherwise pull the latest changes
if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

print("Working in:", os.getcwd())





Download dataset

In [ ]:

"Download the UCMerced Land Use dataset if not already present. "
"The dataset will be saved in the 'ucmdata' directory. "

import os
import zipfile
import subprocess
import shutil
if not os.path.exists('ucmdata'):
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')

    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
        zip_ref.extractall('UCMImages')

    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
UCM_images_path = "/ucmdata/Images/"
Multilabels_path = "/ucmdata/LandUse_Multilabeled.txt"

print(os.listdir('.'))


Load Packages

In [ ]:
import random, shutil, zipfile
from pathlib import Path
from PIL import Image
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
import torchvision.models as tvm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, average_precision_score, hamming_loss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit



import torchmetrics
import re 

L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

Class list and dataset

Augmentation  

Data split 

In [ ]:
# Full dataloader test 
# Can take a long time (4.5min on Colab GPU T4)
from utils import build_dataloaders, UCMMultilabelDataset


train_loader, val_loader, test_loader, classes, pos_w = build_dataloaders(
    root_dir = "ucmdata",
    label_file = "LandUse_Multilabeled.txt",
    image_size = (224, 224),
    batch_size = 32,
    num_workers = 2,
    val_frac = 0.15,
    test_frac = 0.15,
    seed = 42,
    image_ext = ".tif",
)

# Create a csv output of class distribution
train_labels = torch.cat([labels for _, labels in train_loader]).numpy()
val_labels = torch.cat([labels for _, labels in val_loader]).numpy()
test_labels = torch.cat([labels for _, labels in test_loader]).numpy()

print("Train class freq:", train_labels.sum(axis=0))
print("Val   class freq:", val_labels.sum(axis=0))
print("Test  class freq:", test_labels.sum(axis=0))

Initialize WANDB

Hyperparameters 

In [ ]:
# Define hyperparameters
PRETRAINED_MODEL = "resnet50"
NUM_CLASSES = len(classes)
MAX_EPOCHS = 25
EARLYSTOPPING_EPOCHS = 4
LR = 1e-4
WEIGHT_DECAY = 1e-5
run_count = 3


Load and train model

In [ ]:
def build_resnet50(num_classes):
    weights = tvm.ResNet50_Weights.IMAGENET1K_V2
    model = tvm.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

backbone = build_resnet50(num_classes=NUM_CLASSES)
n_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"ResNet-50 trainable parameters: {n_params:,}  |  output logits: {NUM_CLASSES}")

RESNET Head

In [ ]:
class LitResNetMultilabel(L.LightningModule):
    def __init__(self, model, num_classes, lr=1e-4, weight_decay=1e-4,
                 max_epochs=15, threshold=0.5, pos_weight=None):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        pos_weight = torch.as_tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None

        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.threshold = threshold

        mk = dict(task="multilabel", num_labels=num_classes, threshold=threshold)
        self.train_f1 = torchmetrics.F1Score(average="macro", **mk)
        self.val_f1   = torchmetrics.F1Score(average="macro", **mk)
        self.test_f1  = torchmetrics.F1Score(average="macro", **mk)
        self.val_acc  = torchmetrics.Accuracy(average="macro", **mk)
        self.test_acc = torchmetrics.Accuracy(average="macro", **mk)
        self.val_map  = torchmetrics.AveragePrecision(task="multilabel", num_labels=num_classes, average="macro")
        self.test_map = torchmetrics.AveragePrecision(task="multilabel", num_labels=num_classes, average="macro")


    def forward(self, x):
        return self.model(x)

    def _step(self, batch):
        imgs, labels = batch          # labels: float tensor (B, C)
        logits = self(imgs)
        loss = self.criterion(logits, labels)
        return loss, logits, labels

    def training_step(self, batch, batch_idx):
        loss, logits, labels = self._step(batch)
        self.train_f1.update(logits, labels.int())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_f1",   self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, logits, labels = self._step(batch)
        self.val_f1.update(logits, labels.int())
        self.val_acc.update(logits, labels.int())
        self.val_map.update(logits, labels.int())
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_f1",   self.val_f1,  on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc",  self.val_acc, on_step=False, on_epoch=True)
        self.log("val_map",  self.val_map, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        loss, logits, labels = self._step(batch)
        self.test_f1.update(logits, labels.int())
        self.test_acc.update(logits, labels.int())
        self.test_map.update(logits, labels.int())
        self.log("test_loss", loss, on_step=False, on_epoch=True)
        self.log("test_f1",   self.test_f1,  on_step=False, on_epoch=True)
        self.log("test_acc",  self.test_acc, on_step=False, on_epoch=True)
        self.log("test_map",  self.test_map, on_step=False, on_epoch=True)

    def predict_step(self, batch, batch_idx):
        imgs, labels = batch
        logits = self(imgs)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= self.threshold).int()
        return {"probs": probs, "preds": preds, "labels": labels.int()}

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(),
                                      lr=self.hparams.lr,
                                      weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.max_epochs)
        return [optimizer], [scheduler]

Training

In [ ]:
from utils import LightningModuleMultilabel 

lit_model = LightningModuleMultilabel(model=backbone, num_classes=NUM_CLASSES,
                                      lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS, pos_weight=pos_w)
# Callbacks 
checkpoint_cb = ModelCheckpoint(
    dirpath="outputs/checkpoints",
    filename="resnet50-multilabel-best-{epoch:02d}-{val_f1:.4f}",
    monitor="val_f1", mode="max", save_top_k=1, save_weights_only=True,
)

early_stopping = EarlyStopping(
            monitor="val_f1",  
            patience=EARLYSTOPPING_EPOCHS,        # Number of epochs with no improvement
            mode="max",        # Maximize F1 score
            verbose=None
            )

# Logger to plot later
csv_logger = CSVLogger(save_dir="outputs/logs", name="resnet50_multilabel")

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices="auto",
    callbacks=[early_stopping, checkpoint_cb],
    logger=csv_logger,
    log_every_n_steps=10,
)
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)



Final classification on test data

In [ ]:
# Evaluate using the best checkpoint
trainer.test(lit_model, dataloaders=test_loader, ckpt_path="best")

best_path = checkpoint_cb.best_model_path
print(f"Best checkpoint: {best_path}")
lit_model = LitResNetMultilabel.load_from_checkpoint(best_path, model=lit_model.model)

In [ ]:

# Collect probs, preds and ground-truth for downstream analysis
preds_out = trainer.predict(lit_model, dataloaders=test_loader)
test_probs  = torch.cat([b["probs"]  for b in preds_out], dim=0).cpu().numpy()
test_preds  = torch.cat([b["preds"]  for b in preds_out], dim=0).cpu().numpy()
test_labels = torch.cat([b["labels"] for b in preds_out], dim=0).cpu().numpy()

In [ ]:
from utils import compute_test_metrics

# Usage
metrics = compute_test_metrics(test_preds, test_labels, test_probs)

# Print metrics
print(f"\nTest accuracy   : {metrics['accuracy']:.4f}")
print(f"Test macro F1   : {metrics['macro_f1']:.4f}")
print(f"Test micro F1   : {metrics['micro_f1']:.4f}")
print(f"Test samples F1 : {metrics['samples_f1']:.4f}")
print(f"Test macro mAP  : {metrics['macro_map']:.4f}")
print(f"Hamming loss    : {metrics['hamming_loss']:.4f}")
print(f"Exact-match acc : {metrics['subset_acc']:.4f}")



Visualize

In [ ]:
# Usage
from utils import plot_training_curves
plot_training_curves(csv_logger, model_name="ResNet-50 multilabel", 
                     save_path=None)

In [ ]:
from utils import plot_per_class_metrics

# Usage
fig, ax, summary_df = plot_per_class_metrics(
    test_labels=test_labels,
    test_preds=test_preds,
    test_probs=test_probs,
    classes=classes,
    macro_f1=metrics['macro_f1'],
    macro_map=metrics['macro_map'],
    model_name="ResNet-50 multilabel",
    save_path="outputs/resnet50_multilabel_per_class.png",
    csv_output="outputs/resnet50_multilabel_per_class.csv"
)

Examples of images

In [ ]:

from utils import plot_prediction_grid
fig = plot_prediction_grid(
    test_preds=test_preds,
    test_labels=test_labels,
    test_probs=test_probs,
    classes=classes,
    test_loader=test_loader,
    root_dir="ucmdata",
    label_file="LandUse_Multilabeled.txt",
    n_show=9,
    seed=4,
    save_path="outputs/resnet50_multilabel_predictions_grid.png"
)